# HDS Clinical Foundations Data Validation

### Key Functions:
1. **raw_source_ndjson_data_stats**: Computes statistics for NDJSON files within a root directory.
2. **get_bronze_clinical_fhir_statistics**: Computes statistics on the number of records and distinct ids per resource type in the Bronze ClinicalFHIR Delta table.
3. **get_lakehouse_statistics**: Computes the statistics of all tables in specified Silver Lakehouse databases.
4. **compare_clinical_fhir_to_lakehouse**: Compares ClinicalFHIR statistics to Lakehouse statistics to identify discrepancies.
5. **compare_raw_source_to_bronze**: Compares statistics between raw NDJSON source data and the Bronze ClinicalFHIR table to identify discrepancies.

In [ ]:
workspace_id = ""
bronze_lakehouse_id = ""
databases = []
destination_lakehouse_id = ""
destination_lakehouse_path = ""
deployment_environment = ""

In [ ]:
root_source_data_path = f'abfss://{workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{bronze_lakehouse_id}/Files/SampleData/Clinical/FHIR-NDJSON/FHIR-HDS/51KSyntheticPatients'
bronze_delta_table_path = f"abfss://{workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{bronze_lakehouse_id}/Tables/ClinicalFhir"

In [ ]:
from pyspark.sql.functions import input_file_name, col, count, countDistinct, coalesce, lit
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, LongType
from concurrent.futures import ThreadPoolExecutor
import unittest
from pyspark.sql import SparkSession
import io
import logging
import sempy.fabric as fabric
import xmlrunner

class ClinicalFoundationsIngestionTests(unittest.TestCase):

    def __init__(self, methodName='runTest', spark=None, workspace_id = None, bronze_lakehouse_id = None, databases = []):
        super().__init__(methodName)
        logging.basicConfig()
        self.logger = logging.getLogger("LOG")
        self.spark = spark
        self.workspace_id = workspace_id
        self.bronze_lakehouse_id = bronze_lakehouse_id
        self.databases = databases

    def raw_source_ndjson_data_stats(self, root_source_data_path) -> DataFrame:
        """
        Function to get statistics on the number of records in NDJSON files within the root source path, recursively.
        
        :param root_source_data_path: Root path to source data.
        :return: A DataFrame containing resourceType and count of records per resourceType.
        """
        # Use Spark to recursively read all NDJSON files from the root source path
        df = self.spark.read.format("json").option("recursiveFileLookup", "true").load(root_source_data_path)
        df = df.filter(input_file_name().endswith('.ndjson'))
        
        # Count the number of records per resource type
        resource_counts_df = df.groupBy("resourceType").agg(
            count("*").alias("record_count")
        )
        
        # Calculate total rows and print summary
        total_rows = df.count()
        total_files = len(df.select(input_file_name()).distinct().collect())
        
        print(f'Total number of records across all NDJSON files: {total_rows}')
        print(f'Total number of NDJSON files in sample data: {total_files}')
        
        print('\nResource Type Counts:')
        resource_counts = resource_counts_df.collect()
        for row in resource_counts:
            print(f'{row["resourceType"]}: {row["record_count"]}')
        
        return resource_counts_df

    def get_bronze_clinical_fhir_statistics(self, delta_table_path) -> DataFrame:
        """
        Function to get statistics on the number of records per resourceType and the number of distinct ids per resourceType.
        
        :param delta_table_path: Path to the Delta table.
        :return: A DataFrame containing resourceType, total count, and distinct id count.
        """
        # Read the entire Delta table
        df = self.spark.read.format("delta").load(delta_table_path)
        
        # Group by resourceType and calculate counts and distinct counts
        bronze_stats_df = df.groupBy(col("resourceType")).agg(
            count("*").alias("count"),
            countDistinct("id").alias("distinct_count")
        )
        
        return bronze_stats_df

    def get_lakehouse_statistics(self, databases) -> DataFrame:
        """
        Function to get statistics on the number of records in each table within a list of databases.
        
        :param databases: A list of database names to analyze.
        :return: A DataFrame containing database name, table name, and record count.
        """
        # Define schema explicitly to avoid empty schema issues
        schema = StructType([
            StructField("database", StringType(), True),
            StructField("resource", StringType(), True),
            StructField("silver_record_count", LongType(), True)
        ])
        
        # Initialize an empty list to hold the results
        stats_list = []

        def process_table(database, table):
            try:
                if table.tableType == 'MANAGED':  # Ensure it's a table and not a view or temporary table
                    table_name = table.name
                    # Load the table and count the number of records using DataFrame API, minimizing shuffles
                    record_count = self.spark.read.table(f"{database}.{table_name}").count()
                    # Append the result to the stats list
                    stats_list.append((database, table_name, record_count))
            except Exception as e:
                print(f"Error processing table {table.name} in database {database}: {str(e)}")

        for database in databases:
            try:
                # Set the current database
                self.spark.catalog.setCurrentDatabase(database)
                
                # Get a list of tables in the current database
                tables = self.spark.catalog.listTables(database)
                
                # Process tables in parallel
                with ThreadPoolExecutor() as executor:
                    for table in tables:
                        executor.submit(process_table, database, table)

            except Exception as e:
                print(f"Error processing database {database}: {str(e)}")

        # Handle the case where stats_list is empty
        if not stats_list:
            print("No tables found or no records available.")
            return None

        # Convert the results to a DataFrame
        lakehouse_stats_df = self.spark.createDataFrame(stats_list, schema)
        
        # Order the DataFrame by record count in descending order
        lakehouse_stats_df = lakehouse_stats_df.orderBy(col("silver_record_count").desc())
        
        return lakehouse_stats_df

    def compare_clinical_fhir_to_lakehouse(self, clinical_fhir_df, lakehouse_stats_df) -> DataFrame:
        """
        Function to compare the distinct count of records in the ClinicalFHIR table with the count of records
        in each of the tables in the lakehouse.
        
        :param clinical_fhir_df: DataFrame containing ClinicalFHIR statistics.
        :param lakehouse_stats_df: DataFrame containing Lakehouse statistics.
        :return: A DataFrame containing resourceType, ClinicalFHIR distinct count, Silver record count, total count in ClinicalFHIR, and the difference.
        """
        # Join the DataFrames on resourceType
        comparison_df = clinical_fhir_df.join(
            lakehouse_stats_df, 
            clinical_fhir_df.resourceType == lakehouse_stats_df.resource, 
            "outer"
        ).select(
            coalesce(clinical_fhir_df.resourceType, lakehouse_stats_df.resource).alias("resource"), 
            coalesce(col("distinct_count"), lit(0)).alias("clinical_fhir_distinct_count"), 
            coalesce(col("count"), lit(0)).alias("clinical_fhir_total_count"),
            coalesce(col("silver_record_count"), lit(0)).alias("silver_record_count"),
            (coalesce(col("distinct_count"), lit(0)) - coalesce(col("silver_record_count"), lit(0))).alias("discrepancy_count")
        )
        
        # Check for discrepancies and print details if any
        discrepancies_df = comparison_df.filter(col("discrepancy_count") != 0)
        if discrepancies_df.count() > 0:
            print("Discrepancies found between ClinicalFHIR and Lakehouse tables:")
            display(discrepancies_df)
        else:
            print("No discrepancies found between ClinicalFHIR and Lakehouse tables.")

        # Order the DataFrame by record count in descending order
        comparison_df = comparison_df.orderBy(col("clinical_fhir_distinct_count").desc(), col("silver_record_count").desc())
        return comparison_df

    def compare_raw_source_to_bronze(self, raw_source_df, bronze_df) -> DataFrame:
        """
        Function to compare the distinct count of records in the raw source NDJSON data with the count of records
        in the bronze ClinicalFHIR table.
        
        :param raw_source_df: DataFrame containing raw source NDJSON statistics.
        :param bronze_df: DataFrame containing Bronze ClinicalFHIR statistics.
        :return: A DataFrame containing resourceType, raw source record count, bronze record count, and the difference.
        """
        # Join the DataFrames on resourceType
        comparison_df = raw_source_df.join(
            bronze_df, 
            raw_source_df.resourceType == bronze_df.resourceType, 
            "outer"
        ).select(
            coalesce(raw_source_df.resourceType, bronze_df.resourceType).alias("resource"),
            coalesce(col("record_count"), lit(0)).alias("raw_source_record_count"),
            coalesce(col("count"), lit(0)).alias("bronze_record_count"),
            (coalesce(col("record_count"), lit(0)) - coalesce(col("count"), lit(0))).alias("discrepancy_count")
        )
        
        # Check for discrepancies and print details if any
        discrepancies_df = comparison_df.filter(col("discrepancy_count") != 0)
        if discrepancies_df.count() > 0:
            print("Discrepancies found between raw source NDJSON data and Bronze ClinicalFHIR table:")
            display(discrepancies_df)
        else:
            print("No discrepancies found between raw source NDJSON data and Bronze ClinicalFHIR table.")
        
        return comparison_df
    
    def test_raw_to_bronze_data_ingestion(self):
        
        exclusion_list = ["RiskAssessment"]
        root_source_data_path = f'abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/SampleData/Clinical/FHIR-NDJSON/FHIR-HDS/51KSyntheticPatients'
        source_data_stats_df = self.raw_source_ndjson_data_stats(root_source_data_path).cache()
        bronze_statistics_df = self.get_bronze_clinical_fhir_statistics(bronze_delta_table_path).cache()
        source_to_bronze_comparison_df = self.compare_raw_source_to_bronze(source_data_stats_df, bronze_statistics_df)
        
        # Iterate over the rows and perform the checks
        rows = source_to_bronze_comparison_df.collect()
        for row in rows:
            resource = row['resource']
            discrepancy_count = row['discrepancy_count']
            
            if resource not in exclusion_list:
                assert discrepancy_count == 0, f"Difference is not 0 for resource_type {resource}"
            else:
                print(f"Resource type {resource} is in the exclusion list, skipping check.")

    def test_bronze_to_silver_data_ingestion(self):
        
        exclusion_list = ["DocumentReferenceContent"]
        root_source_data_path = f'abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/SampleData/Clinical/FHIR-NDJSON/FHIR-HDS/51KSyntheticPatients'
        bronze_delta_table_path = f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Tables/ClinicalFhir"

        bronze_statistics_df = self.get_bronze_clinical_fhir_statistics(bronze_delta_table_path).cache()
        lakehouse_stats_df = self.get_lakehouse_statistics(databases).cache()
        comparison_df = self.compare_clinical_fhir_to_lakehouse(bronze_statistics_df, lakehouse_stats_df)
        display(comparison_df)

        rows = comparison_df.collect()

        # Iterate over the rows and perform the checks
        for row in rows:
            resource = row['resource']
            discrepancy_count = row['discrepancy_count']
            
            if resource not in exclusion_list:
                assert discrepancy_count == 0, f"Difference is not 0 for resource_type {resource}"
            else:
                print(f"Resource type {resource} is in the exclusion list, skipping check.")

def run_tests_and_write_output(spark):
    
    # Load and run tests
    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(ClinicalFoundationsIngestionTests)

    # Inject parameters / context to the tests
    for test in suite:
        test.spark = spark
        test.workspace_id = workspace_id
        test.bronze_lakehouse_id = bronze_lakehouse_id
        test.databases = databases

    # Write XML test report to stream, decode after completion
    write_stream = io.BytesIO()
    xmlrunner.XMLTestRunner(output=write_stream, verbosity=3).run(suite)
    xml_output = write_stream.getvalue().decode('utf-8')

    # Write report to lakehouse
    mssparkutils.fs.put(f"abfss://{workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{destination_lakehouse_id}/{destination_lakehouse_path}", xml_output, overwrite=True)
    return xml_output

report = run_tests_and_write_output(spark)